# 프롬프트 기본 개념과 구조


## 실습 환경 설정
먼저 필요한 라이브러리와 API 키를 설정합니다.

In [ ]:
# 필요한 패키지 설치 (최초 1회만)
# !pip install langchain langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# API KEY 정보 로드
load_dotenv()

# ChatOpenAI 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")

# 문자열 출력 파서
output_parser = StrOutputParser()

print("✅ 환경 설정 완료!")
print(f"사용 모델: gpt-4o-mini")

In [ ]:
response = model.invoke("안녕하세요?")
print(response.content)

### 랭스미스 프로젝트 추적 설정 (선택사항)
실습 결과를 LangSmith에서 추적하려면 아래 셀을 실행하세요.

In [ ]:
# LangSmith 프로젝트 설정
# .env 파일에 LANGCHAIN_API_KEY, LANGCHAIN_TRACING_V2 등이 이미 설정되어 있다고 가정

# 프로젝트 이름 설정
os.environ["LANGCHAIN_PROJECT"] = "03-prompt-basic"

print("✅ LangSmith 프로젝트 추적 활성화")
print(f"프로젝트: {os.environ.get('LANGCHAIN_PROJECT')}")

---


## 01 언어모델의 기본 동작 원리
### ▶︎ 언어모델이 확률로 다음 토큰을 예측해 답을 생성하는 기본 메커니즘을 이해합니다.


#### Transformer 아키텍처 이해하기
현대의 대규모 언어 모델(LLM)은 대부분 Transformer 구조를 기반으로 동작합니다.

![Transformer 모델 구조](images/01-transformer-basics/01-Transformer-architecture1.png)

> 출처: [Attention Is All You Need (Vaswani et al., 2017)](https://arxiv.org/pdf/1706.03762)

---

### 언어 모델의 핵심 동작 과정
언어 모델이 텍스트를 생성하는 과정은 크게 4단계로 이루어집니다.

- **1단계**: 
  - 토큰화·임베딩: 입력 문장은 토큰으로 쪼개져 벡터(임베딩)로 변환되고, 위치 정보가 더해져 모델에 들어갑니다.

- **2단계**: 
  - 변환(트랜스포머 디코더 블록 반복): “마스크드 멀티헤드 어텐션 → 피드포워드 네트워크 → 잔차연결·정규화”가 여러 층 반복되며, 이전 토큰들만 보게(마스킹) 설계되어 다음 토큰을 예측할 표현을 만듭니다.

- **3단계**: 
  - 어텐션 핵심: 쿼리(Q)·키(K)·값(V)로 문맥을 요약하는 주의를 여러 머리(헤드)로 병렬 계산해, 다양한 관점의 문맥 정보를 결합합니다.

- **4단계**: 다음 토큰 예측·생성 루프: 마지막에 선형변환+소프트맥스로 “다음 토큰 분포”를 만들고, 샘플링/탑-k 등으로 한 토큰을 선택·추가합니다(그다음 토큰도 같은 과정을 반복).

이 과정을 이해하면 왜 프롬프트의 구조와 문맥이 중요한지 알 수 있습니다.

---

## 02 입력과 출력 구조 이해하기
### ▶︎ 입력(프롬프트)과 출력(응답)의 구성요소·흐름을 도식화해 이해합니다.

![입력과 출력구조 이해하기2](./images/02-input-output/02-Prompt-structure1.png)

**Chain of Thought Prompting**

![입력과 출력구조 이해하기2](./images/02-input-output/02-Prompt-structure2.png)

**Self-Consistency Prompting**

![입력과 출력구조 이해하기3](./images/02-input-output/02-Prompt-structure3.png)

**Tree-of-Thought Prompting**

![입력과 출력구조 이해하기4](./images/02-input-output/02-Prompt-structure4.png)

**Basic Prompts**

![입력과 출력구조 이해하기5](./images/02-input-output/02-Prompt-structure5.png)


### ▶︎ 실습문제를 풀어봅시다. (1번~2번)

#### 1. 실습문제
<a id="section1"></a>
✏️ **실습 목표:**
- 프롬프트에 변수를 활용하여 다양한 입력을 생성할 수 있습니다.

<hr>

**실습 문제**
```bash
Prompt (input): 
현재 {$한국}의 대통령은 

Output: 
```

<hr>

아래의 프롬프트를 얼마든지 **수정** 하여 테스트 해볼 수 있습니다.
- `{country}`는 변수이므로 수정하지 마세요.
- `model` 역시 수정하여 테스트 가능합니다.

<hr>

In [ ]:
# 프롬프트를 입력하세요.
template = """
현재 {country}의 대통령은?
"""

In [ ]:
# 프롬프트 템플릿을 이용하여 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(template)

# ChatOpenAI 챗모델을 초기화합니다. - 모델 변경 가능
model = ChatOpenAI(model="gpt-4o-mini")

# 문자열 출력 파서를 초기화합니다.
output_parser = StrOutputParser()

# 프롬프트, 모델, 출력 파서를 순서대로 연결하는 체인을 만듭니다.
chain = prompt | model | output_parser

In [ ]:
# 완성된 체인을 실행하여 결과를 출력합니다.
# 프롬프트에 있는 변수 country 를 대한민국으로 설정하여 실행합니다.
print(chain.invoke({"country": "대한민국"}))

In [ ]:
# 이번에는 'country' 를 '미국'으로 설정하여 실행합니다.
print(chain.invoke({"country": "미국"}))

#### 2. 실습문제
<a id="section2"></a>
✏️ **실습 목표:**
- 프롬프트에 두 개 이상의 변수를 활용하여 다양한 입력을 생성할 수 있습니다.
- 여러 변수를 포함한 프롬프트 템플릿을 작성하고 적용하는 방법을 이해합니다.

<hr>

**실습 문제**
```bash
Prompt (input): 
현재 {$한국}의 대통령의 주요 {$정책}은 

Output: 
```

<hr>

In [ ]:
# 프롬프트를 입력하세요.
template = """
{country}의 대통령의 주요 {policy}은?
"""

In [ ]:
# 프롬프트 템플릿에 넣을 'country'와 'policy' 변수 값을 넣어 결과를 확인합니다.
input = {"country": "미국", "policy": "경제"}

# 위에서 입력한 입력값을 프롬프트 템플릿에 적용해 최종 프롬프트를 확인해봅니다.
formatted_prompt = prompt.format(country=input["country"], policy=input["policy"])
print("✅ 최종 프롬프트:", formatted_prompt)

In [ ]:
# 새로운 프롬프트를 위해 체인을 재구성합니다.
# 프롬프트 템플릿을 이용하여 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(template)

# ChatOpenAI 챗모델을 초기화합니다. - 모델 변경 가능
model = ChatOpenAI(model="gpt-4o-mini")

# 문자열 출력 파서를 초기화합니다.
output_parser = StrOutputParser()

# 프롬프트, 모델, 출력 파서를 순서대로 연결하는 체인을 만듭니다.
chain = prompt | model | output_parser

In [ ]:
# LLM에 프롬프트를 전달하여 답변을 출력합니다.
print("✅ 답변:\n", chain.invoke(input))

In [ ]:
# 이번에는 'country' 를 '한국', 'policy' 를 '교통법'으로 설정하여 실행합니다.
input = {"country": "한국", "policy": "교통법"}
print("✅ 답변: \n", chain.invoke(input))

## 03 프롬프트 설계 기본 요소
### ▶︎ 목적·역할·제약·형식 등 프롬프트의 필수 요소를 정확히 정의해 쓸 수 있습니다.


- **지시 (Instructions)**: 모델이 수행할 특정 작업 또는 지시

- **맥락 (Context)**: 모델이 수행할 특정 작업에 대한 참고 지식이나 배경

- **데이터(Input Data)**: 답변에 참고 할 예시

- **출력 지시문 (Output Indicator)**:응답 형식이나 결과 포맷

![프롬프트 type A](./images/03-prompt-elements/03-Prompt-type.png)


### ▶︎ 실습문제를 풀어봅시다. (3번~6번)


#### 3. 실습문제
<a id="section3"></a>
✏️ **실습 목표:**
- 감정 분석(Sentiment Analysis) 프롬프트를 작성할 때, **지시문**, **맥락**, **입력 데이터**, **출력 지시문** 등 설계 요소를 명확히 구분하여 적용할 수 있습니다.

<hr>

**실습 문제**
```bash
"그 음식 맛이 그저 그랬어” 

문장의 sentiment analysis 를 하는 프롬프트를 작성해보세요.
작성 후 프롬프트의 구성 요소로 나눠 라벨을 붙여주세요.  
```

<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```python
template = """
Sentiment Analysis 를 해야 해. 
아래 텍스트를 긍정, 중립, 부정 중에서 구분해줘. 

Text: {text}
"""
```

</details>

In [ ]:
# 프롬프트를 입력하세요.
template = """
Sentiment Analysis 를 해야 해. 
아래 텍스트를 긍정, 중립, 부정 중에서 구분해줘. 

Text: {text}
"""

In [ ]:
# 새로운 프롬프트를 위해 체인을 재구성합니다.
# 프롬프트 템플릿을 이용하여 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(template)

# ChatOpenAI 챗모델을 초기화합니다. - 모델 변경 가능
model = ChatOpenAI(model="gpt-4o-mini")

# 문자열 출력 파서를 초기화합니다.
output_parser = StrOutputParser()

# 프롬프트, 모델, 출력 파서를 순서대로 연결하는 체인을 만듭니다.
chain = prompt | model | output_parser

In [ ]:
input = {"text": "그 음식 맛이 그저 그랬어."}
print("✅ 답변:\n", chain.invoke(input))

#### 4. 실습문제
<a id="section4"></a>
✏️ **실습 목표:**
- 프롬프트에서 **출력 형식**(한 단어, 목록, JSON 등)을 명확히 지정하여 원하는 형태의 답변을 얻을 수 있습니다.

<hr>

**실습 문제**
```bash
"그 음식 맛이 그저 그랬어” 

문장의 sentiment analysis 를 하는 프롬프트를 작성해보세요.
단, 실습 3과 달리 프롬프트의 답변의 형식을 한 단어로만 나오도록 형식을 지정해주세요. 
```

<hr>

<details>
<summary>🔽 정답 프롬프트 보기</summary>

```python
template = """
Sentiment Analysis 를 해야 해. 
아래 텍스트를 긍정, 중립, 부정 중에서 한 단어로 구분해줘. 

Text: {text}
{{sentiment}}: {{분석결과}} 
"""
```
</details>

In [ ]:
# 프롬프트를 입력하세요.
template = """
Sentiment Analysis 를 해야 해. 
아래 텍스트를 긍정, 중립, 부정 중에서 한 단어로 구분해줘. 

Text: {text} 
{{sentiment}}: {{분석결과}} 
"""

In [ ]:
# 새로운 프롬프트를 위해 체인을 재구성합니다.
# 프롬프트 템플릿을 이용하여 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(template)

# ChatOpenAI 챗모델을 초기화합니다. - 모델 변경 가능
model = ChatOpenAI(model="gpt-4o-mini")

# 문자열 출력 파서를 초기화합니다.
output_parser = StrOutputParser()

# 프롬프트, 모델, 출력 파서를 순서대로 연결하는 체인을 만듭니다.
chain = prompt | model | output_parser

In [ ]:
input = {"text": "그 음식 맛이 그저 그랬어."}
print("✅ 답변:\n", chain.invoke(input))

#### 5. 실습문제
<a id="section5"></a>
✏️ **실습 목표:**
- 다양한 텍스트에 대해 감정 분석을 정확하고 일관성있는 프롬프트를 작성할 수 있습니다.

<hr>

**실습 문제**
```bash
분석 할 문장을 추가했습니다.
프롬프트의 결과물이 일관적으로 정답을 도출하도록 작성해주세요. 
```
**입력 데이터**
```bash
1. 영화가 의외로 꽤 괜찮았어.
2. 서비스가 많이 실망스럽더라
3. 행사는 그냥 무난했어
4. 새 폰 배터리 오래가더라
5. 공연이 생각보다 괜찮더라
6. 품질이 기대에 못미쳤어
7. 업데이트 후 큰 차이는 없었어
8. 배송이 너무 느려서 짜증났어  
9. 회의는 예정대로 진행 될 거야. 
10. 직원들이 정말 친절해서 좋았어. 
```

<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```python
template = """
Sentiment Analysis 를 해야 해. 
아래 텍스트를 긍정, 중립, 부정 중에서 한 단어로 구분해줘. 

Text: {text} 
{{sentiment}}: {{분석결과}} 
"""
```

</details>

In [ ]:
# 프롬프트를 입력하세요.
template = """
Sentiment Analysis 를 해야 해. 
아래 텍스트를 긍정, 중립, 부정 중에서 한단어로 구분해줘. 

Text: {text} 
{{sentiment}}: {{분석결과}} 
"""

In [ ]:
# 새로운 프롬프트를 위해 체인을 재구성합니다.
# 프롬프트 템플릿을 이용하여 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(template)

# ChatOpenAI 챗모델을 초기화합니다. - 모델 변경 가능
model = ChatOpenAI(model="gpt-4o-mini")

# 문자열 출력 파서를 초기화합니다.
output_parser = StrOutputParser()

# 프롬프트, 모델, 출력 파서를 순서대로 연결하는 체인을 만듭니다.
chain = prompt | model | output_parser

In [ ]:
input = {
    "text": """
1. 영화가 의외로 꽤 괜찮았어.
2. 서비스가 많이 실망스럽더라
3. 행사는 그냥 무난했어
4. 새 폰 배터리 오래가더라
5. 공연이 생각보다 괜찮더라
6. 품질이 기대에 못미쳤어
7. 업데이트 후 큰 차이는 없었어
8. 배송이 너무 느려서 짜증났어  
9. 회의는 예정대로 진행 될 거야. 
10. 직원들이 정말 친절해서 좋았어."""
}
print("✅ 답변:\n", chain.invoke(input))

#### 6. 실습문제
<a id="section6"></a>
✏️ **실습 목표:**
- 프롬프트를 활용하여 텍스트 데이터의 노이즈(특수문자, 이모지, 중복 공백 등)를 제거하고 데이터 클리닝 작업을 자동화 할 수 있습니다.

<hr>

**실습 문제**
```bash
데이터 클리닝을 위한 프롬프트를 작성해보세요. 
기호, 숫자, 이모지, 중복문자·공백/특수문자 를 제거해주세요.  
```

**입력 데이터**
```json
{"text":"영화가 의외로 꽤 괜찮았어 ㅎㅎ (7/10) #추천?","label":"긍정"}
{"text":"서비!스가 많이 실망스러웠어... ㅜㅜ @desk","label":"부정"}
{"text":"행사는 그냥 무난하게 끝났어 — 13:45 종료.","label":"중립"}
{"text":"새 폰 배터리 오래가더라🔋 x2, 24h 유지!","label":"긍정"}
{"text":"배송이 너—무 느려서 짜증났어;; 3일 지연...","label":"부정"}
{"text":"회의는   예정대로  진행됐어.  ver.2  (회의실 B-1)","label":"중립"}
{"text":"공연이 생각보다 재미있었어!!!  티켓 2장  : )","label":"긍정"}
{"text":"품질이 기대에 못 미쳤어,,, 5만원값 못함;;","label":"부정"}
{"text":"업데이트 후 큰 차이는 없었어  v1.0.3→v1.0.4","label":"중립"}
{"text":"직원들이 친절해서 좋았어 ^_^  ⭐ 4/5","label":"긍정"}

```
<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```python
template = """
Sentiment Analysis 를 해야 해. 
아래 텍스트를 긍정, 중립, 부정 중에서 한 단어로 구분해줘. 

Text: {text} 
{{sentiment}}: {{분석결과}} 
"""
```

</details>

In [ ]:
# 프롬프트를 입력하세요.
template = """
데이터 클리닝을 위해 기호, 숫자, 이모지, 중복문자·공백/특수문자를 제거해주세요.  

Text: {text} 
"""

In [ ]:
# 새로운 프롬프트를 위해 체인을 재구성합니다.
# 프롬프트 템플릿을 이용하여 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(template)

# ChatOpenAI 챗모델을 초기화합니다. - 모델 변경 가능
model = ChatOpenAI(model="gpt-4o-mini")

# 문자열 출력 파서를 초기화합니다.
output_parser = StrOutputParser()

# 프롬프트, 모델, 출력 파서를 순서대로 연결하는 체인을 만듭니다.
chain = prompt | model | output_parser

In [ ]:
input = {
    "text": """
{"text":"영화가 의외로 꽤 괜찮았어 ㅎㅎ (7/10) #추천?","label":"긍정"}
{"text":"서비!스가 많이 실망스러웠어... ㅜㅜ @desk","label":"부정"}
{"text":"행사는 그냥 무난하게 끝났어 — 13:45 종료.","label":"중립"}
{"text":"새 폰 배터리 오래가더라🔋 x2, 24h 유지!","label":"긍정"}
{"text":"배송이 너—무 느려서 짜증났어;; 3일 지연...","label":"부정"}
{"text":"회의는   예정대로  진행됐어.  ver.2  (회의실 B-1)","label":"중립"}
{"text":"공연이 생각보다 재미있었어!!!  티켓 2장  : )","label":"긍정"}
{"text":"품질이 기대에 못 미쳤어,,, 5만원값 못함;;","label":"부정"}
{"text":"업데이트 후 큰 차이는 없었어  v1.0.3→v1.0.4","label":"중립"}
{"text":"직원들이 친절해서 좋았어 ^_^  ⭐ 4/5","label":"긍정"}
"""
}
print("✅ 답변:\n", chain.invoke(input))

## 04 프롬프트 템플릿
### ▶︎ 반복 과제를 위한 재사용 가능한 프롬프트 템플릿을 설계·적용합니다.

> 정의 : 고정된 지시문 + 변수 자리(예: {{목표}}, {{톤}}, {{입력}})로 이뤄진 재사용 가능한 프롬프트 
작업마다 변수만 바꿔 일관성 있게 고품질 프롬프트 제작 가능 

**사용 목적**
1. 일관성/재현성: 팀·시스템 간 같은 형식으로 요청 → 결과 변동 줄이기
2. 효율/확장성: 반복 작업(요약·분류·추출)을 빠르게 대량 처리
3. 품질 향상: 모범 사례(명확성·형식 지시·예시 포함)를  템플릿화. 

**방법** 
> Write variables like this: {{VARIABLE_NAME}} 

<hr>

**Use prompt templates and variables**
- Fixed content: Static instructions or context that remain constant across multiple interactions
- Variable content: Dynamic elements that change with each request or conversation, such as:
  - User inputs
  - Retrieved content for Retrieval-Augmented Generation (RAG)
  - Conversation context such as user account history
  - System-generated data such as tool use results fed in from other independent calls to Claude

**When to use prompt templates and variables**
- **Consistency**: Ensure a consistent structure for your prompts across multiple interactions
- **Efficiency**: Easily swap out variable content without rewriting the entire prompt
- **Testability**: Quickly test different inputs and edge cases by changing only the variable portion
- **Scalability**: Simplify prompt management as your application grows in complexity
- **Version control**: Easily track changes to your prompt structure over time by keeping tabs only on the core part of your prompt, separate from dynamic inputs


### ▶︎ 실습문제를 풀어봅시다. (7번~8번)

#### 7. 실습문제
<a id="section7"></a>
✏️ **실습 목표:**
- 프롬프트 템플릿을 활용하여 다양한 언어 간 번역 작업을 수행할 수 있습니다.

<hr>

**실습 문제**
```bash
Translate this text from English to Korean: {text}
```

<details>
<summary>🔽 입력 데이터</summary>

```json

OpenAI may be reversing course on how it approaches copyright and intellectual property in its new video app Sora.

Prior to Sora’s launch this week, The Wall Street Journal reported that OpenAI had been telling Hollywood studios and agencies that they needed to explicitly opt out if they didn’t want their IP to be included in Sora-generated videos.

Despite being invite-only, the app quickly climbed to the top of the App Store charts. Sora’s most distinctive feature may be its “cameos,” where users can upload their biometric data to see their digital likeness featured in AI-generated videos.

At the same time, users also seem to delight in flouting copyright laws by creating videos with popular, studio-owned characters. In some cases, those characters might even criticize the company’s approach to copyright, for example in videos where Pikachu and SpongeBob interact with deepfakes of OpenAI CEO Sam Altman.

In a blog post published Friday, Altman said the company is already planning two changes to Sora, first by giving copyright holders “more granular control over generation of characters, similar to the opt-in model for likeness but with additional controls.”

The key word here appears to be “opt-in,” suggesting that OpenAI will stop users from creating videos with copyrighted characters unless studios and others rightsholders have actually given Sora permission to do so.

“We are hearing from a lot of rightsholders who are very excited for this new kind of ‘interactive fan fiction’ and think this new kind of engagement will accrue a lot of value to them, but want the ability to specify how their characters can be used (including not at all),” Altman said. Even with this new approach, Altman acknowledged there are likely to be “some edge cases of generations that get through that shouldn’t.”

The second change he mentioned is some unspecified form of video monetization. The company previously said its only plan for monetization was to charge users to create extra videos during periods of high demand, and Altman’s blog post seems to elaborate on that idea by acknowledging “we are going to have to somehow make money for video generation.” He also suggesting the revenue could be shared with rightsholders.

“Our hope is that the new kind of engagement is even more valuable than the revenue share, but of course we … want both to be valuable.”

```
</details>

<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```python
template = """
아래 텍스트를 영어에서 한글로 번역해줘.

Text: {text} 
"""
```

</details>

In [ ]:
# 프롬프트를 입력하세요.
template = """
아래 텍스트를 영어에서 한글로 번역해줘.

Text: {text} 
"""

In [ ]:
# 새로운 프롬프트를 위해 체인을 재구성합니다.
# 프롬프트 템플릿을 이용하여 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(template)

# ChatOpenAI 챗모델을 초기화합니다. - 모델 변경 가능
model = ChatOpenAI(model="gpt-4o-mini")

# 문자열 출력 파서를 초기화합니다.
output_parser = StrOutputParser()

# 프롬프트, 모델, 출력 파서를 순서대로 연결하는 체인을 만듭니다.
chain = prompt | model | output_parser

In [ ]:
input = {
    "text": """

OpenAI may be reversing course on how it approaches copyright and intellectual property in its new video app Sora.

Prior to Sora’s launch this week, The Wall Street Journal reported that OpenAI had been telling Hollywood studios and agencies that they needed to explicitly opt out if they didn’t want their IP to be included in Sora-generated videos.

Despite being invite-only, the app quickly climbed to the top of the App Store charts. Sora’s most distinctive feature may be its “cameos,” where users can upload their biometric data to see their digital likeness featured in AI-generated videos.

At the same time, users also seem to delight in flouting copyright laws by creating videos with popular, studio-owned characters. In some cases, those characters might even criticize the company’s approach to copyright, for example in videos where Pikachu and SpongeBob interact with deepfakes of OpenAI CEO Sam Altman.

In a blog post published Friday, Altman said the company is already planning two changes to Sora, first by giving copyright holders “more granular control over generation of characters, similar to the opt-in model for likeness but with additional controls.”

The key word here appears to be “opt-in,” suggesting that OpenAI will stop users from creating videos with copyrighted characters unless studios and others rightsholders have actually given Sora permission to do so.

“We are hearing from a lot of rightsholders who are very excited for this new kind of ‘interactive fan fiction’ and think this new kind of engagement will accrue a lot of value to them, but want the ability to specify how their characters can be used (including not at all),” Altman said. Even with this new approach, Altman acknowledged there are likely to be “some edge cases of generations that get through that shouldn’t.”

The second change he mentioned is some unspecified form of video monetization. The company previously said its only plan for monetization was to charge users to create extra videos during periods of high demand, and Altman’s blog post seems to elaborate on that idea by acknowledging “we are going to have to somehow make money for video generation.” He also suggesting the revenue could be shared with rightsholders.

“Our hope is that the new kind of engagement is even more valuable than the revenue share, but of course we … want both to be valuable.”

"""
}
print("✅ 답변:\n", chain.invoke(input))

#### 8. 실습문제
<a id="section8"></a>
✏️ **실습 목표:**
- 주어진 맥락(Context)을 바탕으로 질문에 답변하는 Q&A 프롬프트를 작성할 수 있습니다.

<hr>

**실습 문제**
```bash
다음 템플릿을 사용해보세요. 

Context: {context}

Question: {question}

Give a concise answer based on the context. If the answer isn’t explicitly stated, reply: “The provided context does not contain sufficient information to answer this question.” 

한국어: 문맥을 바탕으로 간결하게 답하세요. 답이 문맥에 명시되어 있지 않으면 다음과 같이 응답하세요: “제공된 문맥에는 이 질문에 답하기에 충분한 정보가 없습니다.

Answer:
```

<details>
<summary>🔽 입력 데이터</summary>

```json

OpenAI may be reversing course on how it approaches copyright and intellectual property in its new video app Sora.

Prior to Sora’s launch this week, The Wall Street Journal reported that OpenAI had been telling Hollywood studios and agencies that they needed to explicitly opt out if they didn’t want their IP to be included in Sora-generated videos.

Despite being invite-only, the app quickly climbed to the top of the App Store charts. Sora’s most distinctive feature may be its “cameos,” where users can upload their biometric data to see their digital likeness featured in AI-generated videos.

At the same time, users also seem to delight in flouting copyright laws by creating videos with popular, studio-owned characters. In some cases, those characters might even criticize the company’s approach to copyright, for example in videos where Pikachu and SpongeBob interact with deepfakes of OpenAI CEO Sam Altman.

In a blog post published Friday, Altman said the company is already planning two changes to Sora, first by giving copyright holders “more granular control over generation of characters, similar to the opt-in model for likeness but with additional controls.”

The key word here appears to be “opt-in,” suggesting that OpenAI will stop users from creating videos with copyrighted characters unless studios and others rightsholders have actually given Sora permission to do so.

“We are hearing from a lot of rightsholders who are very excited for this new kind of ‘interactive fan fiction’ and think this new kind of engagement will accrue a lot of value to them, but want the ability to specify how their characters can be used (including not at all),” Altman said. Even with this new approach, Altman acknowledged there are likely to be “some edge cases of generations that get through that shouldn’t.”

The second change he mentioned is some unspecified form of video monetization. The company previously said its only plan for monetization was to charge users to create extra videos during periods of high demand, and Altman’s blog post seems to elaborate on that idea by acknowledging “we are going to have to somehow make money for video generation.” He also suggesting the revenue could be shared with rightsholders.

“Our hope is that the new kind of engagement is even more valuable than the revenue share, but of course we … want both to be valuable.”

```
</details>

<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```python
template = """
Context: {context}

Question: {question}

Give a concise answer based on the context. If the answer isn’t explicitly stated, reply: “The provided context does not contain sufficient information to answer this question.” 

한국어: 문맥을 바탕으로 간결하게 답하세요. 답이 문맥에 명시되어 있지 않으면 다음과 같이 응답하세요: “제공된 문맥에는 이 질문에 답하기에 충분한 정보가 없습니다.

Answer:
"""
```

</details>

영어버전

In [ ]:
# 프롬프트를 입력하세요.
english_template = """
Context: {context}

Question: {question}

Give a concise answer based on the context. If the answer isn’t explicitly stated, reply: “The provided context does not contain sufficient information to answer this question.” 

Answer:
"""

한국어 버전

In [ ]:
# 프롬프트를 입력하세요.
korean_template = """
Context: {context}

Question: {question}

문맥을 바탕으로 간결하게 답하세요. 답이 문맥에 명시되어 있지 않으면 다음과 같이 응답하세요: “제공된 문맥에는 이 질문에 답하기에 충분한 정보가 없습니다.

Answer:
"""

In [ ]:
# 프롬프트 템플릿을 이용하여 프롬프트를 생성합니다.
ko_prompt = PromptTemplate.from_template(korean_template)
en_prompt = PromptTemplate.from_template(english_template)
# ChatOpenAI 챗모델을 초기화합니다. - 모델 변경 가능
model = ChatOpenAI(model="gpt-4o-mini")

# 문자열 출력 파서를 초기화합니다.
output_parser = StrOutputParser()

# 프롬프트, 모델, 출력 파서를 순서대로 연결하는 체인을 만듭니다.
ko_chain = ko_prompt | model | output_parser
en_chain = en_prompt | model | output_parser

In [ ]:
# 질문1: 샘알트만은 소라의 성공에 기뻐했나요?
input1 = {
    "question": "Question 1. 샘알트만은 소라의 성공에 기뻐했나요?",
    "context": """
OpenAI may be reversing course on how it approaches copyright and intellectual property in its new video app Sora.

Prior to Sora’s launch this week, The Wall Street Journal reported that OpenAI had been telling Hollywood studios and agencies that they needed to explicitly opt out if they didn’t want their IP to be included in Sora-generated videos.

Despite being invite-only, the app quickly climbed to the top of the App Store charts. Sora’s most distinctive feature may be its “cameos,” where users can upload their biometric data to see their digital likeness featured in AI-generated videos.

At the same time, users also seem to delight in flouting copyright laws by creating videos with popular, studio-owned characters. In some cases, those characters might even criticize the company’s approach to copyright, for example in videos where Pikachu and SpongeBob interact with deepfakes of OpenAI CEO Sam Altman.

In a blog post published Friday, Altman said the company is already planning two changes to Sora, first by giving copyright holders “more granular control over generation of characters, similar to the opt-in model for likeness but with additional controls.”

The key word here appears to be “opt-in,” suggesting that OpenAI will stop users from creating videos with copyrighted characters unless studios and others rightsholders have actually given Sora permission to do so.

“We are hearing from a lot of rightsholders who are very excited for this new kind of ‘interactive fan fiction’ and think this new kind of engagement will accrue a lot of value to them, but want the ability to specify how their characters can be used (including not at all),” Altman said. Even with this new approach, Altman acknowledged there are likely to be “some edge cases of generations that get through that shouldn’t.”

The second change he mentioned is some unspecified form of video monetization. The company previously said its only plan for monetization was to charge users to create extra videos during periods of high demand, and Altman’s blog post seems to elaborate on that idea by acknowledging “we are going to have to somehow make money for video generation.” He also suggesting the revenue could be shared with rightsholders.

“Our hope is that the new kind of engagement is even more valuable than the revenue share, but of course we … want both to be valuable.”
""",
}


# 질문2: 샘 알트만은 사용자가 생성한 동영상의 수익화에 대해 찬성하고 있죠?
input2 = {
    "question": "Question 2.  샘 알트만은 사용자가 생성한 동영상의 수익화에 대해 찬성하고 있죠?",
    "context": """
OpenAI may be reversing course on how it approaches copyright and intellectual property in its new video app Sora.

Prior to Sora’s launch this week, The Wall Street Journal reported that OpenAI had been telling Hollywood studios and agencies that they needed to explicitly opt out if they didn’t want their IP to be included in Sora-generated videos.

Despite being invite-only, the app quickly climbed to the top of the App Store charts. Sora’s most distinctive feature may be its “cameos,” where users can upload their biometric data to see their digital likeness featured in AI-generated videos.

At the same time, users also seem to delight in flouting copyright laws by creating videos with popular, studio-owned characters. In some cases, those characters might even criticize the company’s approach to copyright, for example in videos where Pikachu and SpongeBob interact with deepfakes of OpenAI CEO Sam Altman.

In a blog post published Friday, Altman said the company is already planning two changes to Sora, first by giving copyright holders “more granular control over generation of characters, similar to the opt-in model for likeness but with additional controls.”

The key word here appears to be “opt-in,” suggesting that OpenAI will stop users from creating videos with copyrighted characters unless studios and others rightsholders have actually given Sora permission to do so.

“We are hearing from a lot of rightsholders who are very excited for this new kind of ‘interactive fan fiction’ and think this new kind of engagement will accrue a lot of value to them, but want the ability to specify how their characters can be used (including not at all),” Altman said. Even with this new approach, Altman acknowledged there are likely to be “some edge cases of generations that get through that shouldn’t.”

The second change he mentioned is some unspecified form of video monetization. The company previously said its only plan for monetization was to charge users to create extra videos during periods of high demand, and Altman’s blog post seems to elaborate on that idea by acknowledging “we are going to have to somehow make money for video generation.” He also suggesting the revenue could be shared with rightsholders.

“Our hope is that the new kind of engagement is even more valuable than the revenue share, but of course we … want both to be valuable.”
""",
}

In [ ]:
# 영어 버전 답변
print(f"✅ {input1['question']}\n- English Answer:", en_chain.invoke(input1))
# 한국어 버전 답변
print(f"✅ {input1['question']}\n- Korean Answer:", ko_chain.invoke(input1))

print("--------------------------------")

# 영어 버전 답변
print(f"✅ {input2['question']}\n- English Answer:", en_chain.invoke(input2))
# 한국어 버전 답변
print(f"✅ {input2['question']}\n- Korean Answer:", ko_chain.invoke(input2))

## 05 프롬프트 10가지 작성 전략
### ▶︎ 반복 과제를 위한 재사용 가능한 프롬프트 템플릿을 설계·적용합니다.


## 06 LLM의 한계 이해하기
### ▶︎ 환각·편향·지식 한계 등 LLM의 제약을 진단하고 프롬프트로 대응 방법을 적용합니다.


## 07 주요 파라미터 이해하기
### ▶︎ temperature·top-p·max tokens 등 주요 파라미터를 조정해 출력 품질을 최적화합니다.


## 08 프롬프트 작성 전에 준비해야 할 것 (최종 목표, 맥락, 평가 기준)
### ▶︎ 최종 목표·맥락·평가 기준을 사전에 정립해 측정 가능한 결과를 설계합니다.